In [6]:

from pathlib import Path
import time
import pandas as pd
from ultralytics import YOLO

# ==================================================
# 1. ĐƯỜNG DẪN MODEL
# Thứ tự khai báo quyết định tên folder kết quả
# ==================================================
MODELS = {
    "YOLO26m": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo26m\train-railway-yolo26m.pt"
    ),
    "YOLO11s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo11s\train-railway-yolo11s.pt"
    ),
}

# ==================================================
# 2. ĐƯỜNG DẪN DATASET
# ==================================================
DATASETS = {
    "roboflow_crossing": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing"
    ),
    "rcd_ground": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_ground"
    ),
    "rcd_elevated": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated"
    ),
}

# ==================================================
# 3. CẤU HÌNH ĐÁNH GIÁ
# ==================================================
IMGSZ = 640
BATCH = 16
CONF = 0.25
IOU = 0.60

# ==================================================
# 4. TẠO FOLDER KẾT QUẢ THEO TÊN MODEL
# Ví dụ: 26m_11s, 26s_11s, 11s_26m
# ==================================================
BASE_OUTPUT_DIR = Path(
    r"C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results"
)

def get_model_short_name(model_name):
    return model_name.removeprefix("YOLO")

model_order = list(MODELS.keys())
folder_name = "_".join(
    get_model_short_name(name) for name in model_order
)

OUTPUT_DIR = BASE_OUTPUT_DIR / folder_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục lưu kết quả:", OUTPUT_DIR)

# ==================================================
# 5. TÌM FILE DATA.YAML
# ==================================================
def find_data_yaml(dataset_dir):
    for filename in ("data.yaml", "data.yml"):
        yaml_path = dataset_dir / filename
        if yaml_path.is_file():
            return yaml_path

    candidates = sorted(
        list(dataset_dir.rglob("data.yaml"))
        + list(dataset_dir.rglob("data.yml"))
    )

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"Không tìm thấy data.yaml trong {dataset_dir}"
    )

# ==================================================
# 6. KIỂM TRA ĐƯỜNG DẪN
# ==================================================
for model_name, model_path in MODELS.items():
    if not model_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy model {model_name}: {model_path}"
        )

DATA_YAMLS = {}

for dataset_name, dataset_dir in DATASETS.items():
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy dataset: {dataset_dir}"
        )

    DATA_YAMLS[dataset_name] = find_data_yaml(dataset_dir)

print("\nCác file YAML được sử dụng:")
for name, path in DATA_YAMLS.items():
    print(f"{name}: {path}")

# ==================================================
# 7. ĐÁNH GIÁ CÁC MODEL TRÊN TỪNG DATASET
# ==================================================
results = []

for model_name, model_path in MODELS.items():

    print("\n" + "=" * 70)
    print(f"ĐANG LOAD MODEL: {model_name}")
    print("=" * 70)

    model = YOLO(str(model_path))

    for dataset_name, yaml_path in DATA_YAMLS.items():

        print(f"\nModel: {model_name}")
        print(f"Dataset: {dataset_name}")
        print("-" * 70)

        run_name = f"{model_name}_{dataset_name}"

        start = time.perf_counter()

        metrics = model.val(
            data=str(yaml_path),
            split="val",
            imgsz=IMGSZ,
            batch=BATCH,
            conf=CONF,
            iou=IOU,
            plots=True,
            project=str(OUTPUT_DIR),
            name=run_name,
            exist_ok=True,
            verbose=True,
        )

        elapsed = time.perf_counter() - start
        box = metrics.box

        result = {
            "Model": model_name,
            "Dataset": dataset_name,
            "mAP50-95": float(box.map),
            "mAP50": float(box.map50),
            "mAP75": float(box.map75),
            "Precision": float(box.mp),
            "Recall": float(box.mr),
            "Validation_time_sec": round(elapsed, 2),
        }

        results.append(result)

        print("\nKẾT QUẢ:")
        print(f"mAP50-95 : {box.map:.4f}")
        print(f"mAP50    : {box.map50:.4f}")
        print(f"mAP75    : {box.map75:.4f}")
        print(f"Precision: {box.mp:.4f}")
        print(f"Recall   : {box.mr:.4f}")
        print(f"Thời gian: {elapsed:.2f} giây")

        # Lưu tiến độ sau mỗi lượt đánh giá
        pd.DataFrame(results).to_csv(
            OUTPUT_DIR / "comparison_progress.csv",
            index=False,
            encoding="utf-8-sig",
        )

# ==================================================
# 8. XUẤT BẢNG TỔNG HỢP
# ==================================================
df = pd.DataFrame(results)

summary_path = OUTPUT_DIR / "comparison_summary.csv"

df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 100)
print("BẢNG SO SÁNH TỔNG HỢP")
print("=" * 100)
print(df.to_string(index=False))

print("\nĐã lưu bảng tổng hợp tại:")
print(summary_path)

# ==================================================
# 9. MODEL TỐT HƠN THEO TỪNG DATASET
# ==================================================
print("\n" + "=" * 100)
print("MODEL TỐT HƠN THEO mAP50-95 TRÊN TỪNG DATASET")
print("=" * 100)

for dataset_name in DATASETS:
    subset = df[df["Dataset"] == dataset_name]

    if subset.empty:
        continue

    best = subset.loc[subset["mAP50-95"].idxmax()]

    print(
        f"{dataset_name}: {best['Model']} "
        f"(mAP50-95 = {best['mAP50-95']:.4f})"
    )

print(f"\nToàn bộ kết quả nằm trong:\n{OUTPUT_DIR}")


Thư mục lưu kết quả: C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results\26m_11s

Các file YAML được sử dụng:
roboflow_crossing: C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\data.yaml
rcd_ground: C:\Users\anhla\Downloads\dataset_testv3\rcd_ground\data.yaml
rcd_elevated: C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated\data.yaml

ĐANG LOAD MODEL: YOLO26m

Model: YOLO26m
Dataset: roboflow_crossing
----------------------------------------------------------------------
Ultralytics 8.4.165  Python-3.10.11 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO26m summary (fused): 130 layers, 20,352,536 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1799.3343.7 MB/s, size: 361.2 KB)
val: Scanning C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\labels.cache... 120 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120  0.0s
                 Class     Images  Instances      Box(P          R      

In [7]:

from pathlib import Path
import time
import pandas as pd
from ultralytics import YOLO

# ==================================================
# 1. ĐƯỜNG DẪN MODEL
# Thứ tự khai báo quyết định tên folder kết quả
# ==================================================
MODELS = {
    "YOLO26s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo26s\train-railway-yolo26s.pt"
    ),
    "YOLO11s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo11s\train-railway-yolo11s.pt"
    ),
}

# ==================================================
# 2. ĐƯỜNG DẪN DATASET
# ==================================================
DATASETS = {
    "roboflow_crossing": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing"
    ),
    "rcd_ground": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_ground"
    ),
    "rcd_elevated": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated"
    ),
}

# ==================================================
# 3. CẤU HÌNH ĐÁNH GIÁ
# ==================================================
IMGSZ = 640
BATCH = 16
CONF = 0.25
IOU = 0.60

# ==================================================
# 4. TẠO FOLDER KẾT QUẢ THEO TÊN MODEL
# Ví dụ: 26m_11s, 26s_11s, 11s_26m
# ==================================================
BASE_OUTPUT_DIR = Path(
    r"C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results"
)

def get_model_short_name(model_name):
    return model_name.removeprefix("YOLO")

model_order = list(MODELS.keys())
folder_name = "_".join(
    get_model_short_name(name) for name in model_order
)

OUTPUT_DIR = BASE_OUTPUT_DIR / folder_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục lưu kết quả:", OUTPUT_DIR)

# ==================================================
# 5. TÌM FILE DATA.YAML
# ==================================================
def find_data_yaml(dataset_dir):
    for filename in ("data.yaml", "data.yml"):
        yaml_path = dataset_dir / filename
        if yaml_path.is_file():
            return yaml_path

    candidates = sorted(
        list(dataset_dir.rglob("data.yaml"))
        + list(dataset_dir.rglob("data.yml"))
    )

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"Không tìm thấy data.yaml trong {dataset_dir}"
    )

# ==================================================
# 6. KIỂM TRA ĐƯỜNG DẪN
# ==================================================
for model_name, model_path in MODELS.items():
    if not model_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy model {model_name}: {model_path}"
        )

DATA_YAMLS = {}

for dataset_name, dataset_dir in DATASETS.items():
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy dataset: {dataset_dir}"
        )

    DATA_YAMLS[dataset_name] = find_data_yaml(dataset_dir)

print("\nCác file YAML được sử dụng:")
for name, path in DATA_YAMLS.items():
    print(f"{name}: {path}")

# ==================================================
# 7. ĐÁNH GIÁ CÁC MODEL TRÊN TỪNG DATASET
# ==================================================
results = []

for model_name, model_path in MODELS.items():

    print("\n" + "=" * 70)
    print(f"ĐANG LOAD MODEL: {model_name}")
    print("=" * 70)

    model = YOLO(str(model_path))

    for dataset_name, yaml_path in DATA_YAMLS.items():

        print(f"\nModel: {model_name}")
        print(f"Dataset: {dataset_name}")
        print("-" * 70)

        run_name = f"{model_name}_{dataset_name}"

        start = time.perf_counter()

        metrics = model.val(
            data=str(yaml_path),
            split="val",
            imgsz=IMGSZ,
            batch=BATCH,
            conf=CONF,
            iou=IOU,
            plots=True,
            project=str(OUTPUT_DIR),
            name=run_name,
            exist_ok=True,
            verbose=True,
        )

        elapsed = time.perf_counter() - start
        box = metrics.box

        result = {
            "Model": model_name,
            "Dataset": dataset_name,
            "mAP50-95": float(box.map),
            "mAP50": float(box.map50),
            "mAP75": float(box.map75),
            "Precision": float(box.mp),
            "Recall": float(box.mr),
            "Validation_time_sec": round(elapsed, 2),
        }

        results.append(result)

        print("\nKẾT QUẢ:")
        print(f"mAP50-95 : {box.map:.4f}")
        print(f"mAP50    : {box.map50:.4f}")
        print(f"mAP75    : {box.map75:.4f}")
        print(f"Precision: {box.mp:.4f}")
        print(f"Recall   : {box.mr:.4f}")
        print(f"Thời gian: {elapsed:.2f} giây")

        # Lưu tiến độ sau mỗi lượt đánh giá
        pd.DataFrame(results).to_csv(
            OUTPUT_DIR / "comparison_progress.csv",
            index=False,
            encoding="utf-8-sig",
        )

# ==================================================
# 8. XUẤT BẢNG TỔNG HỢP
# ==================================================
df = pd.DataFrame(results)

summary_path = OUTPUT_DIR / "comparison_summary.csv"

df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 100)
print("BẢNG SO SÁNH TỔNG HỢP")
print("=" * 100)
print(df.to_string(index=False))

print("\nĐã lưu bảng tổng hợp tại:")
print(summary_path)

# ==================================================
# 9. MODEL TỐT HƠN THEO TỪNG DATASET
# ==================================================
print("\n" + "=" * 100)
print("MODEL TỐT HƠN THEO mAP50-95 TRÊN TỪNG DATASET")
print("=" * 100)

for dataset_name in DATASETS:
    subset = df[df["Dataset"] == dataset_name]

    if subset.empty:
        continue

    best = subset.loc[subset["mAP50-95"].idxmax()]

    print(
        f"{dataset_name}: {best['Model']} "
        f"(mAP50-95 = {best['mAP50-95']:.4f})"
    )

print(f"\nToàn bộ kết quả nằm trong:\n{OUTPUT_DIR}")


Thư mục lưu kết quả: C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results\26s_11s

Các file YAML được sử dụng:
roboflow_crossing: C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\data.yaml
rcd_ground: C:\Users\anhla\Downloads\dataset_testv3\rcd_ground\data.yaml
rcd_elevated: C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated\data.yaml

ĐANG LOAD MODEL: YOLO26s

Model: YOLO26s
Dataset: roboflow_crossing
----------------------------------------------------------------------
Ultralytics 8.4.165  Python-3.10.11 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO26s summary (fused): 120 layers, 9,466,728 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1504.774.9 MB/s, size: 360.3 KB)
val: Scanning C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\labels.cache... 120 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 79.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          

In [8]:

from pathlib import Path
import time
import pandas as pd
from ultralytics import YOLO

# ==================================================
# 1. ĐƯỜNG DẪN MODEL
# Thứ tự khai báo quyết định tên folder kết quả
# ==================================================
MODELS = {
    "YOLO26s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo26s\train-railway-yolo26s.pt"
    ),
    "YOLO11s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo11s\train-railway-yolo11s.pt"
    ),
    "YOLO26m": Path(
        r"C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo26m\train-railway-yolo26m.pt"
    ),
}

# ==================================================
# 2. ĐƯỜNG DẪN DATASET
# ==================================================
DATASETS = {
    "roboflow_crossing": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing"
    ),
    "rcd_ground": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_ground"
    ),
    "rcd_elevated": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated"
    ),
}

# ==================================================
# 3. CẤU HÌNH ĐÁNH GIÁ
# ==================================================
IMGSZ = 640
BATCH = 16
CONF = 0.25
IOU = 0.60

# ==================================================
# 4. TẠO FOLDER KẾT QUẢ
# Chỉ tạo folder con bên trong thư mục có sẵn
# Ví dụ: 26s_11s_26m
# ==================================================
BASE_OUTPUT_DIR = Path(
    r"C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results"
)

def get_model_short_name(model_name):
    return model_name.removeprefix("YOLO")

model_order = list(MODELS.keys())
folder_name = "_".join(
    get_model_short_name(name) for name in model_order
)

OUTPUT_DIR = BASE_OUTPUT_DIR / folder_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("THƯ MỤC LƯU KẾT QUẢ")
print(OUTPUT_DIR)
print("=" * 80)

# ==================================================
# 5. TÌM FILE DATA.YAML
# ==================================================
def find_data_yaml(dataset_dir):
    for filename in ("data.yaml", "data.yml"):
        yaml_path = dataset_dir / filename
        if yaml_path.is_file():
            return yaml_path

    candidates = sorted(
        list(dataset_dir.rglob("data.yaml"))
        + list(dataset_dir.rglob("data.yml"))
    )

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"Không tìm thấy data.yaml trong: {dataset_dir}"
    )

# ==================================================
# 6. KIỂM TRA ĐƯỜNG DẪN
# ==================================================
for model_name, model_path in MODELS.items():
    if not model_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy model {model_name}:\n{model_path}"
        )

DATA_YAMLS = {}

for dataset_name, dataset_dir in DATASETS.items():
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy dataset {dataset_name}:\n{dataset_dir}"
        )

    DATA_YAMLS[dataset_name] = find_data_yaml(dataset_dir)

print("\nCÁC FILE DATA.YAML:")
for dataset_name, yaml_path in DATA_YAMLS.items():
    print(f"{dataset_name}: {yaml_path}")

# ==================================================
# 7. ĐÁNH GIÁ 3 MODEL TRÊN 3 DATASET
# ==================================================
results = []
total_runs = len(MODELS) * len(DATASETS)
current_run = 0

progress_path = OUTPUT_DIR / "comparison_progress.csv"
summary_path = OUTPUT_DIR / "comparison_summary.csv"

for model_name, model_path in MODELS.items():

    print("\n" + "=" * 80)
    print(f"ĐANG LOAD MODEL: {model_name}")
    print(f"Đường dẫn: {model_path}")
    print("=" * 80)

    model = YOLO(str(model_path))

    for dataset_name, yaml_path in DATA_YAMLS.items():
        current_run += 1

        print("\n" + "-" * 80)
        print(f"TIẾN ĐỘ: {current_run}/{total_runs}")
        print(f"MODEL: {model_name}")
        print(f"DATASET: {dataset_name}")
        print("-" * 80)

        run_name = f"{model_name}_{dataset_name}"

        start_time = time.perf_counter()

        metrics = model.val(
            data=str(yaml_path),
            split="val",
            imgsz=IMGSZ,
            batch=BATCH,
            conf=CONF,
            iou=IOU,
            plots=True,
            project=str(OUTPUT_DIR),
            name=run_name,
            exist_ok=True,
            verbose=True,
        )

        elapsed = time.perf_counter() - start_time
        box = metrics.box

        result = {
            "Model": model_name,
            "Dataset": dataset_name,
            "mAP50-95": float(box.map),
            "mAP50": float(box.map50),
            "mAP75": float(box.map75),
            "Precision": float(box.mp),
            "Recall": float(box.mr),
            "Validation_time_sec": round(elapsed, 2),
        }

        results.append(result)

        # In kết quả lượt hiện tại
        print("\nKẾT QUẢ:")
        print(f"mAP50-95 : {box.map:.4f}")
        print(f"mAP50    : {box.map50:.4f}")
        print(f"mAP75    : {box.map75:.4f}")
        print(f"Precision: {box.mp:.4f}")
        print(f"Recall   : {box.mr:.4f}")
        print(f"Thời gian: {elapsed:.2f} giây")

        # Lưu tiến độ sau mỗi lượt đánh giá
        pd.DataFrame(results).to_csv(
            progress_path,
            index=False,
            encoding="utf-8-sig",
        )

        print(f"Đã lưu tiến độ: {progress_path}")

    # Giải phóng model trước khi load model tiếp theo
    del model

# ==================================================
# 8. XUẤT BẢNG TỔNG HỢP
# ==================================================
df = pd.DataFrame(results)

df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 100)
print("BẢNG SO SÁNH TỔNG HỢP")
print("=" * 100)
print(df.to_string(index=False))

print("\nĐã lưu bảng tổng hợp tại:")
print(summary_path)

# ==================================================
# 9. MODEL TỐT NHẤT THEO TỪNG DATASET
# ==================================================
print("\n" + "=" * 100)
print("MODEL TỐT NHẤT THEO mAP50-95 TRÊN TỪNG DATASET")
print("=" * 100)

for dataset_name in DATASETS:
    subset = df[df["Dataset"] == dataset_name]

    if subset.empty:
        continue

    best = subset.loc[subset["mAP50-95"].idxmax()]

    print(
        f"{dataset_name}: {best['Model']} "
        f"(mAP50-95 = {best['mAP50-95']:.4f})"
    )

# ==================================================
# 10. SO SÁNH TRUNG BÌNH TRÊN 3 DATASET
# ==================================================
print("\n" + "=" * 100)
print("ĐIỂM TRUNG BÌNH CỦA MỖI MODEL TRÊN 3 DATASET")
print("=" * 100)

average_df = (
    df.groupby("Model")[
        ["mAP50-95", "mAP50", "mAP75", "Precision", "Recall"]
    ]
    .mean()
    .sort_values("mAP50-95", ascending=False)
)

print(average_df.to_string(float_format=lambda x: f"{x:.4f}"))

average_path = OUTPUT_DIR / "comparison_average.csv"
average_df.to_csv(
    average_path,
    encoding="utf-8-sig",
)

best_overall = average_df.index[0]

print(f"\nModel có mAP50-95 trung bình cao nhất: {best_overall}")
print(f"\nToàn bộ kết quả nằm trong:\n{OUTPUT_DIR}")


THƯ MỤC LƯU KẾT QUẢ
C:\Users\anhla\Downloads\Web_flask\finetune\model_comparison_results\26s_11s_26m

CÁC FILE DATA.YAML:
roboflow_crossing: C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\data.yaml
rcd_ground: C:\Users\anhla\Downloads\dataset_testv3\rcd_ground\data.yaml
rcd_elevated: C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated\data.yaml

ĐANG LOAD MODEL: YOLO26s
Đường dẫn: C:\Users\anhla\Downloads\Web_flask\finetune\train_yolo26s\train-railway-yolo26s.pt

--------------------------------------------------------------------------------
TIẾN ĐỘ: 1/9
MODEL: YOLO26s
DATASET: roboflow_crossing
--------------------------------------------------------------------------------
Ultralytics 8.4.165  Python-3.10.11 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO26s summary (fused): 120 layers, 9,466,728 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 898.0254.2 MB/s, size: 313.0 KB)
val: Scanning C:\Users\anhla\Downloads\